In [ ]:
#####################################################
#
# APLICAR Regresión logística a datos preprocesados con PCA
#
#####################################################
# Deben cargarse los archivos
# - T_train_final_objetivo.csv
# - T_test_final_objetivo.csv"
# - pca_pipe_num.joblib
# - pca_metadata.json
# Devolverá
# expected_columns.json (columnas que deberán tener datos que nunca ha visto)
# modelo_reg_lineal.pkl (modelo ya entrenado)
#####################################################

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, f1_score,
    roc_auc_score, confusion_matrix,
    roc_curve, precision_recall_curve
)

# Lectura
Train = pd.read_csv("T_train_final_objetivo.csv")
Test  = pd.read_csv("T_test_final_objetivo.csv")

# Separar X | y (última columna = objetivo binario 0/1)
X_train = Train.iloc[:, :-1].copy()
y_train = Train.iloc[:, -1].astype(int).to_numpy()

X_test  = Test.iloc[:, :-1].copy()
y_test  = Test.iloc[:, -1].astype(int).to_numpy()

SEP = "___"

In [ ]:
def is_binary_series(s: pd.Series):
    vals = pd.unique(s.dropna())
    return set(vals).issubset({0,1}) or set(vals).issubset({0.0,1.0})

def prefix_of(col: str, sep=SEP):
    return col.split(sep, 1)[0] if sep in col else None

def build_nominal_blocks_by_prefix(X: pd.DataFrame, sep=SEP):
    blocks = {}
    for c in X.columns:
        if sep in c and is_binary_series(X[c]):
            blocks.setdefault(prefix_of(c, sep), []).append(c)
    # respeta orden del CSV
    for k,v in blocks.items():
        blocks[k] = [c for c in X.columns if c in set(v)]
    return blocks


In [ ]:
# 1) --- PRECOMPUTA CON TRAIN ---
blocks = build_nominal_blocks_by_prefix(X_train, SEP)
drop_cols = [cols[0] for cols in blocks.values() if len(cols) >= 2]  # primera de cada bloque

# 2) --- PIPELINE: dropear base de cada bloque + escalar + logística ---
arreglar_despeje = ColumnTransformer(
    transformers=[("drop_nominal_bases", "drop", drop_cols)],
    remainder="passthrough",
    verbose_feature_names_out=False,
    force_int_remainder_cols=False   # ← mantenido para evitar el warning en tu entorno
)

mi_regresion_logistica = Pipeline(steps=[
    ("dropper", arreglar_despeje),
    ("scaler", StandardScaler(with_mean=False)),
    ("logit", LogisticRegression(
        solver="liblinear",
        penalty="l2",
        max_iter=2000,
######################################################################
######################################################################
        class_weight="balanced",   # usa "balanced" si hay desbalance
######################################################################
######################################################################
        fit_intercept=True
    ))
])

# 3) --- FIT
mi_regresion_logistica.fit(X_train, y_train)

# 4) Probabilidades para seguir con α más adelante:
p_train = mi_regresion_logistica.predict_proba(X_train)[:, 1]
p_test  = mi_regresion_logistica.predict_proba(X_test)[:, 1]

In [ ]:
Train["scores"]=p_train
Test["scores"]=p_test

Train.to_csv("T_train_final_objetivo_scores.csv",index=False)
Test.to_csv("T_test_final_objetivo_scores.csv",index=False)

In [ ]:
# =============================================================================
# EVALUACIÓN CON DIFERENTES UMBRALES α
# =============================================================================

def evaluate_thresholds(y_true, probs, thresholds=np.arange(0.1, 1.0, 0.05)):
    """
    Evalúa múltiples métricas para diferentes umbrales de clasificación
    """
    results = []

    for alpha in thresholds:
        y_pred = (probs >= alpha).astype(int)

        # Métricas básicas
        acc = accuracy_score(y_true, y_pred)
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average='binary', zero_division=0
        )

        # Matriz de confusión
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

        # Tasas específicas
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0  # Recall/Sensitivity
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0  # False Positive Rate

        results.append({
            'threshold': alpha,
            'accuracy': acc,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn,
            'tpr': tpr, 'fpr': fpr
        })

    return pd.DataFrame(results)



In [ ]:
# Evaluar en conjunto de TEST
threshold_results = evaluate_thresholds(y_test, p_test)

# Mostrar mejores umbrales por diferentes métricas
print("=== MEJORES UMBRALES EN TEST ===")
print(f"Mejor F1-score: {threshold_results.loc[threshold_results['f1_score'].idxmax()]['threshold']:.3f}")
print(f"Mejor Accuracy: {threshold_results.loc[threshold_results['accuracy'].idxmax()]['threshold']:.3f}")
print(f"Mejor Precision: {threshold_results.loc[threshold_results['precision'].idxmax()]['threshold']:.3f}")
print(f"Mejor Recall: {threshold_results.loc[threshold_results['recall'].idxmax()]['threshold']:.3f}")



In [ ]:
threshold_results

In [ ]:
# =============================================================================
# GRÁFICAS DE EVALUACIÓN
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Métricas vs Umbral
axes[0].plot(threshold_results['threshold'], threshold_results['accuracy'],  label='Accuracy',  marker='o')
axes[0].plot(threshold_results['threshold'], threshold_results['precision'], label='Precision', marker='s')
axes[0].plot(threshold_results['threshold'], threshold_results['recall'],    label='Recall',    marker='^')
axes[0].plot(threshold_results['threshold'], threshold_results['f1_score'],  label='F1-score',  marker='d')
axes[0].set_xlabel('Umbral α')
axes[0].set_ylabel('Score')
axes[0].set_title('Métricas vs Umbral de Clasificación')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Distribución de probabilidades por clase
axes[1].hist(p_test[y_test == 0], bins=30, alpha=0.7, label='Clase 0', color='red',  density=True)
axes[1].hist(p_test[y_test == 1], bins=30, alpha=0.7, label='Clase 1', color='blue', density=True)
axes[1].set_xlabel('Probabilidad predicha')
axes[1].set_ylabel('Densidad')
axes[1].set_title('Distribución de Probabilidades por Clase Real')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# SELECCIÓN DEL UMBRAL ÓPTIMO (múltiples criterios)
# =============================================================================

def find_optimal_threshold(y_true, probs, method='f1'):
    """
    Encuentra el umbral óptimo según diferentes criterios
    """
    if method == 'f1':
        precision, recall, thresholds = precision_recall_curve(y_true, probs)
        # f1_scores tiene len == len(thresholds) + 1
        f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)

        # Tomamos el mejor F1 pero mapeando al umbral correspondiente (excluye el último punto)
        idx = np.argmax(f1_scores[:-1])  # alinear con thresholds
        return thresholds[idx]

    elif method == 'youden':
        fpr, tpr, thresholds = roc_curve(y_true, probs)
        youden = tpr - fpr
        return thresholds[np.argmax(youden)]

    elif method == 'geometric_mean':
        fpr, tpr, thresholds = roc_curve(y_true, probs)
        gmean = np.sqrt(tpr * (1 - fpr))
        return thresholds[np.argmax(gmean)]

    elif method == 'accuracy':
        # Generar múltiples umbrales y calcular accuracy para cada uno
        thresholds = np.arange(0.0, 1.0, 0.01)
        accuracies = []

        for threshold in thresholds:
            # Convertir probabilidades a predicciones binarias
            y_pred = (probs >= threshold).astype(int)
            # Calcular accuracy
            accuracy = accuracy_score(y_true, y_pred)
            accuracies.append(accuracy)

        # Encontrar el umbral que maximiza el accuracy
        return thresholds[np.argmax(accuracies)]

    else:
        return 0.5


# Calcular umbrales óptimos por diferentes métodos
optimal_f1 = find_optimal_threshold(y_test, p_test, 'f1')
optimal_youden = find_optimal_threshold(y_test, p_test, 'youden')
optimal_gmean = find_optimal_threshold(y_test, p_test, 'geometric_mean')
optimal_accuracy = find_optimal_threshold(y_test, p_test, 'accuracy')

print("\n=== UMBRALES ÓPTIMOS SEGÚN DIFERENTES CRITERIOS ===")
print(f"Por F1-score: {optimal_f1:.3f}")
print(f"Por índice de Youden: {optimal_youden:.3f}")
print(f"Por media geométrica: {optimal_gmean:.3f}")
print(f"Por Accuracy: {optimal_accuracy:.3f}")

In [ ]:
# =============================================================================
# EVALUACIÓN FINAL CON UMBRAL ELEGIDO
# =============================================================================

# Elegir umbral (por ejemplo, el que maximiza F1)
##################################################
##################################################
alpha_optimo = optimal_f1
##################################################
##################################################
print(f"\n=== EVALUACIÓN FINAL CON α = {alpha_optimo:.3f} ===")

y_pred_optimo = (p_test >= alpha_optimo).astype(int)

# Métricas finales
acc_final = accuracy_score(y_test, y_pred_optimo)
precision_final, recall_final, f1_final, _ = precision_recall_fscore_support(
    y_test, y_pred_optimo, average='binary', zero_division=0
)

print(f"Accuracy (porcentaje de casos que se han clasificado correctamente):  {acc_final:.3f}")
print(f"Precision (de todas las predicciones positivas, ¿qué proporción son correctas?): {precision_final:.3f}")
print(f"Recall (De todos los positivos reales, ¿qué proporción detectaste?):    {recall_final:.3f}")
print(f"F1-score (precisión y sensibilidad):  {f1_final:.3f}")
#print(f"AUC-ROC:   {roc_auc_score(y_test, p_test):.3f}")

# Matriz de confusión final
cm = confusion_matrix(y_test, y_pred_optimo)
print(f"\nMatriz de confusión:\n{cm}")

In [ ]:
###### Guardado del modelo

import joblib, json, time

# guarda el pipeline completo (dropper + LinearRegression)
joblib.dump(mi_regresion_logistica, "modelo_reg_logistica.pkl")

# guarda el orden/esperado de columnas de entrenamiento
expected_cols = X_train.columns.tolist()
with open("expected_columns.json", "w", encoding="utf-8") as f:
    json.dump({"columns": expected_cols, "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")}, f)

print("Artefactos guardados:", "modelo_reg_logistica.pkl", "expected_columns.json")


In [ ]:
import os, zipfile, glob

# Carpeta destino en tu PC
dst_dir = r"mi_regresion_logistica"
os.makedirs(dst_dir, exist_ok=True)
zip_path = os.path.join(dst_dir, "mi_reg_logistica_artifacts_bundle.zip")

# Archivos que quieres incluir (ajusta si te falta alguno)
candidates = [
    "modelo_reg_logistica.pkl",
    "expected_columns.json",
    "T_train_final_objetivo_scores.csv",
    "T_test_final_objetivo_scores.csv"
]

present = [f for f in candidates if os.path.exists(f)]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for f in present:
        zf.write(f, arcname=os.path.basename(f))

print("ZIP creado en:", zip_path)
print("Incluidos:", present)
